## Spielbaum

[Video](https://youtu.be/_VYbjbDNaVc) - [Video2](https://youtu.be/v8rSRVvUcTY) - [Aufgabenvideo](https://youtu.be/fqlQ2TsTdRM)

Wie kann ein Computer Tic-Tac-Toe, Vier gewinnt oder Schach spielen? Er probiert alle Züge aus, dann alle Antworten
des Gegners, dann wieder alle eigenen Züge und so weiter. So entsteht ein Baum aller möglichen Spielverläufe, der
Spielbaum. In diesem Notebook sehen wir, wie man darin den besten Zug findet (MinMax-Algorithmus), wie man große Teile
des Baums überspringen kann (Alpha-Beta-Algorithmus) und was man tut, wenn der Baum zu groß ist (Bremse).

#### Mehrwegebaum

In einem *Mehrwegebaum* kann jeder Knoten eine variable Anzahl von Kindern besitzen.

<img src='mehrwegebaum.png' width='600'>

#### Spielbaum

Ein *Spielbaum* ist ein Mehrwegebaum mit zwei Typen von Knoten: Maximum-Knoten und Minimum-Knoten. Die Knoten
repräsentieren Spielstellungen, die Kanten zwischen den Knoten repräsentieren Spielzüge.

Der Spielbaum eignet sich für Zwei-Personen-Nullsummenspiele wie Schach, Dame oder Tic-Tac-Toe: Zwei Personen ziehen
abwechselnd, und der Gewinn des einen Spielers ist der Verlust des anderen. Ein Spieler heißt **Max**, er möchte eine
möglichst hohe Bewertung erreichen. Der andere heißt **Min**, er möchte eine möglichst niedrige Bewertung. In den
Max-Knoten ist Max am Zug, in den Min-Knoten Min.

- Der Wert eines Blattes wird bestimmt durch eine statische **Stellungsbewertung**, z.B. 1 für einen Gewinn von Max,
  $-1$ für einen Gewinn von Min und 0 für unentschieden.
- Der Wert eines Max-Knotens ist das Maximum der Werte seiner Kinder.
- Der Wert eines Min-Knotens ist das Minimum der Werte seiner Kinder.

Dahinter steckt die Annahme, dass beide Spieler immer den für sie besten Zug wählen. Der Wert der Wurzel ist dann
das, was Max sicher erreichen kann, egal wie Min spielt.

#### Beispiel

<img src='spielbaum.png' width='900'>

Die Knoten mit ihren Bewertungen:

<img src='spielbaum01.png' width='900'>

Die Werte werden von unten nach oben berechnet. Ein Knoten erhält seinen Wert, sobald alle seine Kinder einen Wert
haben. Wenn man den Baum mit einer Tiefensuche von links nach rechts durchläuft, erhalten die Knoten ihre Bewertung in
dieser Reihenfolge:

    e:2 f:3 b:2 g:4 h:-2 i:2 c:-2 j:1 d:1 a:2
    bester Zug: b

Max wählt den Zug nach b, denn b hat unter den Kindern von a den größten Wert.

### Der MinMax-Algorithmus

Für die Implementierung des MinMax-Algorithmus setzen wir drei Funktionen voraus:

- `nextstates(state)`: die Liste der Folgestellungen,
- `terminal_test(state)`: `True`, wenn die Stellung ein Blatt ist, also das Spiel zu Ende,
- `evaluation(state)`: die Stellungsbewertung.

Mit diesen Funktionen können wir `maximize` und `minimize` schreiben, unabhängig vom konkreten Spiel. Die beiden
Funktionen rufen sich gegenseitig auf, denn nach einem Zug von Max ist Min am Zug und umgekehrt.

In [1]:
inf = float('inf')

def maximize(state):
    '''
    state: Spielstellung, Max ist am Zug
    returns: (st, k), die Folgestellung st, die die höchste Bewertung k hat.
             Ist state ein Blatt, gibt es keine Folgestellung, nur die Bewertung.
    '''
    if terminal_test(state):
        return None, evaluation(state)
    best_st, best_k = None, -inf
    for st in nextstates(state):
        _, k = minimize(st)
        if k > best_k:
            best_st, best_k = st, k
    return best_st, best_k

def minimize(state):
    '''
    state: Spielstellung, Min ist am Zug
    returns: (st, k), die Folgestellung st, die die niedrigste Bewertung k hat
    '''
    if terminal_test(state):
        return None, evaluation(state)
    best_st, best_k = None, inf
    for st in nextstates(state):
        _, k = maximize(st)
        if k < best_k:
            best_st, best_k = st, k
    return best_st, best_k

#### Beispiel-Spielbaum

Beispiel-Spielbäume geben wir mit zwei Dictionaries an: `nxt` enthält zu jedem inneren Knoten die Liste seiner
Kinder, `blatt` die Bewertung der Blätter.

In [2]:
nxt = {'a': list('bc'), 'b': list('de'), 'c': list('fg')}   # Eltern-Kind-Beziehungen
blatt = {'d': 10, 'e': 8, 'f': 4, 'g': 50}                  # Stellungsbewertung

def nextstates(state):
    return nxt[state]

def terminal_test(state):
    return state in blatt

def evaluation(state):
    return blatt[state]

maximize('a')

('b', 8)

<img src='minmax01.png' width='500'>

Reihenfolge der Bewertungen (ohne Blätter): `b:8 c:4 a:8`, bester Zug: b. Obwohl unter c das Blatt mit der höchsten
Bewertung 50 liegt, wählt Max den Zug nach b: Nach c würde Min mit f antworten, und Max bekäme nur 4.

Dasselbe für den Baum aus dem Beispiel oben:

In [3]:
nxt = {'a': list('bcd'), 'b': list('ef'), 'c': list('ghi'), 'd': list('j'), 'e': list('kl'),
       'f': list('mno'), 'g': list('p'), 'h': list('qr'), 'i': list('st'), 'j': list('uvw')}
blatt = {'k': 2, 'l': -4, 'm': -2, 'n': -1, 'o': 3, 'p': 4, 'q': -2, 'r': -5, 's': 2, 't': -1,
         'u': -3, 'v': 1, 'w': -2}
maximize('a')

('b', 2)

### Tic-Tac-Toe

Es gibt mehrere Möglichkeiten, eine Spielstellung zu modellieren.

    x . .
    . o .
    . . .

In [4]:
state = [['x', '.', '.'], ['.', 'o', '.'], ['.', '.', '.']]
state = [[1, 0, 0], [0, -1, 0], [0, 0, 0]]
state = [(0, 0), (1, 1)]
state = [0, 4]                                  # wir wählen diese Modellierung

Eine Spielstellung ist eine Liste von Zahlen. Jede Zahl ist die Nummer eines Feldes, in der Reihenfolge der Züge.

     x . .       0  1  2
     . o .       3  4  5
     . . .       6  7  8

Die Spielstellung oben ist also die Liste `[0, 4]`. Der Max-Spieler (x) beginnt, deshalb gehören die Felder an den
geraden Positionen der Liste zu x, die an den ungeraden zu o. Die Bewertung ist 1, wenn x gewonnen hat, $-1$, wenn o
gewonnen hat, und sonst 0.

In [5]:
gewinn = [{0, 1, 2}, {3, 4, 5}, {6, 7, 8}, {0, 3, 6},
          {1, 4, 7}, {2, 5, 8}, {0, 4, 8}, {2, 4, 6}]

def nextstates(state):
    '''returns: Liste mit möglichen Folgestellungen (also Liste von Listen)'''
    temp = []
    for i in range(9):
        if i not in state:
            temp.append(state + [i])
    return temp

def evaluation(state):
    '''returns: 1, wenn x gewonnen hat, -1, wenn o gewonnen hat, sonst 0'''
    maxzuege = set(state[::2])
    minzuege = set(state[1::2])
    for g in gewinn:
        if g <= maxzuege:
            return 1
        if g <= minzuege:
            return -1
    return 0

def terminal_test(state):
    '''returns: True, wenn jemand gewonnen hat oder alle Felder belegt sind'''
    global blaetter
    w = evaluation(state)
    if w == 1 or w == -1 or len(state) == 9:
        blaetter += 1                  # zählt, wie viele Blätter bewertet werden
        return True
    return False

def show(state):
    '''printed die Stellung'''
    a = ['.'] * 9
    for i in range(len(state)):
        if i % 2 == 0:
            a[state[i]] = 'x'
        else:
            a[state[i]] = 'o'
    for i in range(9):
        if i % 3 == 0:
            print()
        print(a[i], end=' ')
    print()

blaetter = 0
show([0, 4])


x . . 
. o . 
. . . 


`g <= maxzuege` prüft, ob die Menge `g` eine Teilmenge der Felder von x ist, ob x also alle drei Felder einer
Gewinnreihe besetzt hat.

Wie geht Tic-Tac-Toe aus, wenn beide perfekt spielen? Wir lassen den MinMax-Algorithmus den ganzen Spielbaum ab der
leeren Stellung durchsuchen:

In [6]:
blaetter = 0
print(maximize([]))
print('bewertete Blätter:', blaetter)

([0], 0)
bewertete Blätter: 255168


Der Wert ist 0: Bei perfektem Spiel endet Tic-Tac-Toe unentschieden. Dafür wurden 255168 Blätter bewertet, das sind
alle möglichen Spielverläufe. Der Zug `[0]` ist einfach der erste mit dem Wert 0, andere Eröffnungen sind genauso gut.

Jetzt hat x in der Ecke und am Rand gesetzt, o in der Mitte. o ist am Zug und muss die Reihe 0, 1, 2 blockieren:

In [7]:
state = [0, 4, 1]
show(state)
st, k = minimize(state)
show(st)
print('Bewertung:', k)


x x . 
. o . 
. . . 

x x o 
. o . 
. . . 
Bewertung: 0


Mit der folgenden Funktion kann man gegen den Computer spielen. Der Computer ist o und wählt seine Züge mit
`minimize`. Zum Spielen die Raute vor `play()` entfernen.

In [8]:
def showMuster():
    '''Hilfsfunktion, die die Nummerierung der Felder zeigt'''
    for i in range(9):
        if i % 3 == 0:
            print()
        print(i, end=' ')
    print()

def play():
    state = []
    i = 0
    while not terminal_test(state):
        if i % 2 == 0:
            showMuster()
            z = int(input('Eingabe Spieler x: '))
            state.append(z)
        else:
            state, _ = minimize(state)
            show(state)
        i += 1
    show(state)
    if evaluation(state) == 1:
        print('Spieler x hat gewonnen')
    elif evaluation(state) == -1:
        print('Computer hat gewonnen')
    else:
        print('Unentschieden')

# play()

### Der Alpha-Beta-Algorithmus

Der Alpha-Beta-Algorithmus versucht, Teile des Baumes abzuschneiden (pruning), die erkennbar nicht mehr durchsucht
werden müssen. Er liefert dieselbe Bewertung und denselben besten Zug wie der MinMax-Algorithmus, nur schneller.

<img src='alphaBeta01.png' width='800'>

Die Auswertung mit pruning:

<img src='alphaBeta02.png' width='800'>

Warum darf bei c abgeschnitten werden? Max kann über b schon sicher 3 erreichen. Das erste Kind h von c hat den Wert 2,
also wird c höchstens 2. Was unter i und j steht, spielt keine Rolle mehr: Max wird c ohnehin nicht wählen. Bei d ist
es genauso, sobald l mit 2 gefunden ist.

Jedem Knoten wird ein *alpha*- und ein *beta*-Wert mitgegeben, die im Verlauf des Algorithmus aktualisiert werden.
$\alpha$ ist der Wert, den Max auf dem bisherigen Weg schon sicher hat, $\beta$ der Wert, den Min schon sicher hat.
Mit ihnen lässt sich ein Kriterium formulieren, wann das pruning erfolgen kann.

#### Vereinfachte Notation zur Durchführung des Alpha-Beta-Algorithmus

Zu Beginn ist $\alpha = -\infty$ und $\beta = +\infty$.

$\alpha$ und $\beta$ werden von oben nach unten durchgereicht. Bei den Max-Knoten wird $\beta$ nicht verändert, bei den
Min-Knoten wird $\alpha$ nicht verändert. Wir notieren deshalb nur die $\alpha$-Werte bei den Max-Knoten und nur die
$\beta$-Werte bei den Min-Knoten. An den Blättern notieren wir keine $\alpha$- oder $\beta$-Werte.

Der Algorithmus ist eine Tiefensuche, die Abwärts- und Aufwärtsbewegungen macht. Abwärts erben die Kinder ihre
$\alpha$- und $\beta$-Werte von den Großeltern. Wenn aufwärts ein Elternknoten bei einem Kind etwas sieht, was er haben
will, nimmt er es von dem Kind.

Wenn an einer Stelle entdeckt wird, dass $\alpha$ größer oder gleich dem darüberstehenden $\beta$ ist oder dass
$\beta$ kleiner oder gleich dem darüberstehenden $\alpha$ ist, wird gepruned.

Der Auftraggeber des prunings ist der Elternknoten des Knotens, bei dem das pruning erfolgt.

<img src='alphaBeta03.png' width='800'>

Merkregeln:

<img src='kurzform.png' width='600'>

Durchführung des Alpha-Beta-Algorithmus:

<img src='alphaBeta04.png' width='800'>

Reihenfolge des Blattbesuchs, # für pruning:

    k l m # o p # s t u #
    bester Zug: d

#### Der Alpha-Beta-Algorithmus in Python

`maximize` und `minimize` bekommen jetzt zusätzlich `alpha` und `beta`. Findet `maximize` einen Wert, der mindestens
`beta` ist, bricht es die Schleife ab (das ist das pruning). `minimize` bricht ab, wenn es einen Wert findet, der
höchstens `alpha` ist.

Der Parameter `bremse` begrenzt die Tiefe der Suche, dazu mehr im nächsten Abschnitt. Ist er 0, wird die Stellung
bewertet, auch wenn das Spiel noch nicht zu Ende ist.

In [9]:
def maximize(state, alpha=-inf, beta=inf, bremse=100):
    '''
    state: Spielstellung, Max ist am Zug
    returns: (st, k), die Folgestellung st, die die höchste Bewertung k hat
    '''
    if bremse == 0 or terminal_test(state):
        return None, evaluation(state)
    best_st, best_k = None, -inf
    for st in nextstates(state):
        _, k = minimize(st, alpha, beta, bremse - 1)
        if k > best_k:
            best_st, best_k = st, k
        if best_k >= beta:
            break
        if best_k > alpha:
            alpha = best_k
    return best_st, best_k

def minimize(state, alpha=-inf, beta=inf, bremse=100):
    '''
    state: Spielstellung, Min ist am Zug
    returns: (st, k), die Folgestellung st, die die niedrigste Bewertung k hat
    '''
    if bremse == 0 or terminal_test(state):
        return None, evaluation(state)
    best_st, best_k = None, inf
    for st in nextstates(state):
        _, k = maximize(st, alpha, beta, bremse - 1)
        if k < best_k:
            best_st, best_k = st, k
        if best_k <= alpha:
            break
        if best_k < beta:
            beta = best_k
    return best_st, best_k

Wir prüfen das am Baum aus dem Bild oben. Damit wir sehen, welche Blätter besucht werden, merkt sich `evaluation` jedes
Blatt in einer Liste.

In [10]:
nxt = {'a': list('bcd'), 'b': list('ef'), 'c': list('gh'), 'd': list('ij'),
       'e': list('kl'), 'f': list('mn'), 'g': list('op'), 'h': list('qr'), 'i': list('st'), 'j': list('uv')}
blatt = {'k': 4, 'l': 6, 'm': 7, 'n': 9, 'o': 1, 'p': 2, 'q': 0, 'r': 1, 's': 8, 't': 1, 'u': 9, 'v': 2}

def nextstates(state):
    return nxt[state]

def terminal_test(state):
    return state in blatt

def evaluation(state):
    besucht.append(state)
    return blatt[state]

besucht = []
print(maximize('a'))
print('besuchte Blätter:', *besucht)

('d', 8)
besuchte Blätter: k l m o p s t u


Die Blätter n, q, r und v werden nicht besucht, genau dort steht im Bild ein #.

Bei Tic-Tac-Toe ist der Unterschied groß. Wir nehmen wieder die Funktionen von oben:

In [11]:
def nextstates(state):
    temp = []
    for i in range(9):
        if i not in state:
            temp.append(state + [i])
    return temp

def evaluation(state):
    maxzuege = set(state[::2])
    minzuege = set(state[1::2])
    for g in gewinn:
        if g <= maxzuege:
            return 1
        if g <= minzuege:
            return -1
    return 0

def terminal_test(state):
    global blaetter
    w = evaluation(state)
    if w == 1 or w == -1 or len(state) == 9:
        blaetter += 1
        return True
    return False

blaetter = 0
print(maximize([]))
print('bewertete Blätter:', blaetter)

([0], 0)
bewertete Blätter: 7330


Statt 255168 Blättern werden nur 7330 bewertet, das Ergebnis ist dasselbe. Wie viel abgeschnitten wird, hängt von der
Reihenfolge der Züge ab: Wird der beste Zug zuerst untersucht, kann am meisten gepruned werden.

### Die Bremse

Bei Tic-Tac-Toe kann man den ganzen Spielbaum durchsuchen. Bei Vier gewinnt gibt es etwa $4.5 \cdot 10^{12}$
mögliche Stellungen, bei Schach unvorstellbar viele mehr. Dann sucht man nur bis zu einer bestimmten Tiefe, der
**Bremse**. Mit jedem Zug wird `bremse` um 1 kleiner. Bei 0 wird die Stellung bewertet, obwohl das Spiel noch nicht zu
Ende ist.

Dafür braucht man eine Bewertung, die auch unfertige Stellungen einschätzt, also eine **Heuristik**, z.B. bei Vier
gewinnt: Wie viele Reihen hat jeder Spieler schon zu zwei oder drei Steinen gefüllt? Die Qualität des Computerspielers
hängt dann von zwei Dingen ab: wie tief er suchen kann und wie gut die Heuristik ist.

```python
st = ...                             # Ausgangsstellung
print(maximize(st, bremse=8))        # 8 Züge tief suchen
```

### Vier gewinnt (Connect4)

Das Spiel besteht aus 7 Spalten, die jeweils mit bis zu 6 Steinen gefüllt werden können. Wer zuerst vier eigene
Steine waagrecht, senkrecht oder diagonal in einer Reihe hat, gewinnt.

Eine Stellung wird repräsentiert durch eine Liste von Zahlen von 0 bis 41, die jeweils angeben, in welcher Reihenfolge
Steine auf die jeweilige Position gekommen sind.

```
 0  1  2  3  4  5  6
 7  8  9 10 11 12 13
14 15 16 17 18 19 20
21 22 23 24 25 26 27
28 29 30 31 32 33 34
35 36 37 38 39 40 41
```

Der erste Stein ist 'x'. Die Liste `[38, 37, 30]` repräsentiert also die folgende Stellung:

```
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . x . . . .
. . o x . . .
=============
0 1 2 3 4 5 6
```

Im User-Interface gibt man die Nummer der Spalte für den nächsten Stein an.

Die Bewertung sucht in allen Reihen aus vier Feldern nach Mustern: Vier eigene Steine sind ein Gewinn, drei eigene
Steine und ein freies Feld geben viele Punkte, zwei eigene Steine weniger, einer noch weniger. Für x zählen die Punkte
positiv, für o negativ.

In [12]:
gewinn = 10000

def showNr():
    ''' printed die Nummerierung der Felder '''
    for i in range(42):
        if i % 7 == 0:
            print()
        print('{:2d}'.format(i), end=' ')
    print()

def getBoard(state):
    ''' wandelt state in ein board um 
        ein board ist eine Liste mit 42 Zeichen, entweder x, o oder ein Punkt.'''
    tmp = list('.'*42)
    for i in range(len(state)):
        if i % 2 == 0:
            tmp[state[i]] = 'x'
        else:
            tmp[state[i]] = 'o'
    return tmp

def showBoard(board):
    ''' printed board '''
    for i in range(42):
        if i % 7 == 0:
            print()
        print('{:1s}'.format(board[i]), end=' ')
    print()
    print('='*13)
    print('0 1 2 3 4 5 6')

def muster(board):
    tmp = []
    for i in range(0, 42, 7):
        tmp.append(''.join(board[i:i+7]))  # horizontal
    for i in range(7):
        tmp.append(''.join(board[i::7]))   # vertikal

    # diagonale muster nach rechts unten (Schrittweite 8)
    tmp.append(''.join(board[14:39:8]))
    tmp.append(''.join(board[7:40:8]))
    tmp.append(''.join(board[0:41:8]))
    tmp.append(''.join(board[1:42:8]))
    tmp.append(''.join(board[2:35:8]))
    tmp.append(''.join(board[3:28:8]))

    # diagonale muster nach links unten (Schrittweite 6)
    tmp.append(''.join(board[3:22:6]))
    tmp.append(''.join(board[4:29:6]))
    tmp.append(''.join(board[5:36:6]))
    tmp.append(''.join(board[6:37:6]))
    tmp.append(''.join(board[13:38:6]))
    tmp.append(''.join(board[20:39:6]))
    return tmp

def nextstates(state):
    '''
    state: Spielstatus
    returns: Liste mit möglichen Folgestellungen
    '''
    board = getBoard(state)
    tmp = []
    for i in free(board):     # für jeden verfügbaren Platz
        tmp.append(state+[i])
    return tmp

def evaluation(state):
    '''
    state: Spielstellung
    returns int - Bewertung der Spielstellung
    '''
    w = 0
    board = getBoard(state)
    for s in muster(board):
        if 'xxxx' in s:
            return gewinn
        if 'oooo' in s:
            return -gewinn
        if 'x...' in s:
            w += 5
        if '.x..' in s:
            w += 5
        if '..x.' in s:
            w += 5
        if '...x' in s:
            w += 5

        if 'xx..' in s:
            w += 30
        if 'x.x.' in s:
            w += 30
        if 'x..x' in s:
            w += 30
        if '.xx.' in s:
            w += 30
        if '.x.x' in s:
            w += 30
        if '..xx' in s:
            w += 30
        if 'xxx.' in s:
            w += 500
        if 'xx.x' in s:
            w += 500
        if 'x.xx' in s:
            w += 500
        if '.xxx' in s:
            w += 500

        if 'o...' in s:
            w += -5
        if '.o..' in s:
            w += -5
        if '..o.' in s:
            w += -5
        if '...o' in s:
            w += -5

        if 'oo..' in s:
            w += -30
        if 'o.o.' in s:
            w += -30
        if 'o..o' in s:
            w += -30
        if '.oo.' in s:
            w += -30
        if '.o.o' in s:
            w += -30
        if '..oo' in s:
            w += -30
        if 'ooo.' in s:
            w += -500
        if 'oo.o' in s:
            w += -500
        if 'o.oo' in s:
            w += -500
        if '.ooo' in s:
            w += -500
    return w

def terminal_test(state):
    w = evaluation(state)
    return w >= gewinn or w <= -gewinn or len(state) == 42

def colNrs(i):
    '''
    i: int,  0 <= i <= 6
    returns: Liste der Platznummern für Spalte i, von unten nach oben
    '''
    return [35+i-7*r for r in range(0, 6)]

def freeIn(board, i):
    ''' returns: nächster freier Platz in Spalte i '''
    for j in colNrs(i):
        if board[j] == '.':
            return j

def free(board):
    ''' liste der möglichen nächsten positionen '''
    tmp = []
    for i in range(7):
        for j in colNrs(i):
            if board[j] == '.':
                tmp.append(j)
                break
    return tmp

def play():
    ''' Spieler beginnt und spielt x.
    Computer ist Nachziehender und Min-Spieler mit o.
    '''

    BREMSE = 8
    state = []
    board = getBoard(state)
    showBoard(board)

    i = 0
    while not terminal_test(state):

        if i % 2 == 0:
            x = int(input("Eingabe Spieler x (0-6): "))
            state.append(freeIn(board, x))
            board = getBoard(state)
            showBoard(board)
            print("Spieler x hat gezogen",
                  state[-1] % 7, " - Bewertung:", evaluation(state))

        else:
            if tuple(state) in buch:
                state = buch[tuple(state)]
                k = "Buchzug"
            else:
                state, k = minimize(state, bremse=BREMSE)
            board = getBoard(state)
            showBoard(board)
            print("Computer o hat gezogen",
                  state[-1] % 7, " - Bewertung:", evaluation(state), "- min-Wert:", k)

        i = i+1

    if evaluation(state) >= gewinn:
        print("Spieler x hat gewonnen")
    elif evaluation(state) <= -gewinn:
        print("Computer hat gewonnen")
    else:
        print("Unentschieden")

buch = {tuple(): [38], (38,): [38, 31]}   # Eröffnungsbuch
# play()     # Spieler beginnt mit x

`maximize` und `minimize` sind die Funktionen mit Alpha-Beta von oben, sie bleiben unverändert. Im Eröffnungsbuch
`buch` stehen fertige Antworten für die ersten Züge, wie bei Schachprogrammen. Antwortet x auf das leere Brett mit
der mittleren Spalte (38), setzt der Computer darauf (31).

Wie gut sucht der Computer bei verschiedenen Bremsen? Nach den Zügen 38, 31 und 37 ist o am Zug:

In [13]:
import time

state = [38, 31, 37]
showBoard(getBoard(state))
for bremse in [4, 6, 8]:
    t = time.time()
    st, k = minimize(state, bremse=bremse)
    print(f'bremse {bremse}: Spalte {st[-1] % 7}, Bewertung {k}, {time.time() - t:.2f} s')


. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . o . . . 
. . x x . . . 
0 1 2 3 4 5 6
bremse 4: Spalte 4, Bewertung 95, 0.02 s


bremse 6: Spalte 1, Bewertung 115, 0.30 s


bremse 8: Spalte 1, Bewertung 170, 4.70 s


Mit jeder zusätzlichen Stufe dauert die Suche ein Vielfaches länger, denn jede Stellung hat bis zu 7 Folgestellungen.
Mit Bremse 4 setzt der Computer in Spalte 4, mit Bremse 6 und 8 in Spalte 1. Tiefer zu suchen ändert also die Wahl.

Wenn x drei Steine am rechten Rand hat, muss o in Spalte 3 setzen:

In [14]:
state = [39, 32, 40, 33, 41]
showBoard(getBoard(state))
st, k = minimize(state, bremse=4)
showBoard(getBoard(st))
print('Computer setzt in Spalte', st[-1] % 7)


. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . o o . 
. . . . x x x 
0 1 2 3 4 5 6

. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . o o . 
. . . o x x x 
0 1 2 3 4 5 6
Computer setzt in Spalte 3


Zum Spielen gegen den Computer die Raute vor `play()` am Ende der großen Zelle entfernen.